# EquipED DPO Training Template

Before running: open the EquipED admin panel, go to **Training Data**
for the agent you're training, click **Start Training Job**, and paste
the resulting `download_url` and `upload_url` below.

**The download URL is single-use.** If this notebook disconnects or you
need to re-run the fetch cell, re-running it will fail with a 404 --
the token is already spent. Go back to the admin panel and start a new
training job to get a fresh pair of URLs; there is no way to reuse or
extend the old ones.

This notebook contains a complete, self-contained LoRA + DPO training workflow
optimized for Google Colab (including free-tier T4 GPU runtimes). It validates
the downloaded package before any dependency installs or training steps,
fine-tunes the base model with TRL DPOTrainer, bundles a full provenance
training manifest into the adapter, and uploads the verified zip
artifact back to EquipED.

In [ ]:
DOWNLOAD_URL = "PASTE_DOWNLOAD_URL_HERE"
UPLOAD_URL = "PASTE_UPLOAD_URL_HERE"

In [ ]:
import hashlib
import io
import json
import re
from typing import Any
import uuid
import zipfile

import requests

EXPECTED_MANIFEST_VERSION = "equiped.dpo-package.v1"
SUPPORTED_AGENTS = frozenset({"sme", "coordinator", "gad", "itso"})
REQUIRED_PACKAGE_MEMBERS = frozenset({"manifest.json", "pairs.jsonl", "provenance.jsonl"})

REQUIRED_MANIFEST_FIELDS: dict[str, type | tuple[type, ...]] = {
    "manifest_version": str,
    "agent_id": str,
    "model_name": (str, type(None)),
    "response_contract_keys": list,
    "pair_count": int,
    "evaluation_count": int,
    "reviewer_count": int,
    "skipped_counts": dict,
    "pairs_sha256": str,
    "pairs_bytes": int,
    "provenance_sha256": str,
    "provenance_bytes": int,
    "export_timestamp": str,
}

REQUIRED_PAIR_KEYS = frozenset({"prompt", "chosen", "rejected"})

REQUIRED_PROVENANCE_FIELDS: dict[str, type | tuple[type, ...]] = {
    "pair_id": str,
    "generation_id": str,
    "evaluation_id": str,
    "document_id": str,
    "agent_id": str,
    "unit_key": str,
    "model_name": str,
    "response_contract_key": str,
    "response_contract_version": int,
    "criterion_ids": list,
    "reviewer_ids": list,
    "prompt_sha256": str,
    "response_sha256": str,
    "created_at": (str, type(None)),
}
HEX64_RE = re.compile(r"^[0-9a-fA-F]{64}$")

def _is_valid_uuid(val: Any) -> bool:
    try:
        uuid.UUID(str(val))
        return True
    except (ValueError, TypeError, AttributeError):
        return False

response = requests.get(DOWNLOAD_URL)
response.raise_for_status()
package_bytes = response.content

with zipfile.ZipFile(io.BytesIO(package_bytes)) as zf:
    namelist = zf.namelist()
    if len(namelist) != len(set(namelist)):
        raise ValueError("Package archive contains duplicate entries")
    archive_members = frozenset(namelist)
    if archive_members != REQUIRED_PACKAGE_MEMBERS:
        raise ValueError(
            f"Package archive member mismatch: expected exactly {sorted(REQUIRED_PACKAGE_MEMBERS)}, "
            f"got {sorted(archive_members)}"
        )

    raw_manifest_bytes = zf.read("manifest.json")
    try:
        manifest = json.loads(raw_manifest_bytes.decode("utf-8"))
    except Exception as exc:
        raise ValueError(f"Invalid JSON in manifest.json: {exc}") from exc

    if not isinstance(manifest, dict):
        raise ValueError(f"manifest.json must be a JSON object, got {type(manifest).__name__}")

    for field_name, expected_type in REQUIRED_MANIFEST_FIELDS.items():
        if field_name not in manifest:
            raise ValueError(f"manifest.json is missing required field: {field_name!r}")
        value = manifest[field_name]
        if isinstance(expected_type, tuple):
            if not isinstance(value, expected_type):
                type_names = ", ".join(t.__name__ for t in expected_type)
                raise ValueError(
                    f"manifest field {field_name!r} expected one of ({type_names}), got {type(value).__name__}"
                )
        else:
            if not isinstance(value, expected_type):
                raise ValueError(
                    f"manifest field {field_name!r} expected {expected_type.__name__}, got {type(value).__name__}"
                )

    if manifest["manifest_version"] != EXPECTED_MANIFEST_VERSION:
        raise ValueError(
            f"Unsupported manifest_version {manifest['manifest_version']!r}; expected {EXPECTED_MANIFEST_VERSION!r}"
        )

    if manifest["agent_id"] not in SUPPORTED_AGENTS:
        raise ValueError(
            f"Unsupported agent_id {manifest['agent_id']!r}; supported agents: {sorted(SUPPORTED_AGENTS)}"
        )

    raw_pairs_bytes = zf.read("pairs.jsonl")
    raw_provenance_bytes = zf.read("provenance.jsonl")

if len(raw_pairs_bytes) != manifest["pairs_bytes"]:
    raise ValueError(
        f"pairs.jsonl byte length mismatch: manifest specifies {manifest['pairs_bytes']}, "
        f"actual length is {len(raw_pairs_bytes)}"
    )
actual_pairs_sha256 = hashlib.sha256(raw_pairs_bytes).hexdigest()
if actual_pairs_sha256 != manifest["pairs_sha256"]:
    raise ValueError(
        f"pairs.jsonl SHA-256 mismatch: manifest specifies {manifest['pairs_sha256']}, "
        f"actual is {actual_pairs_sha256}"
    )

if len(raw_provenance_bytes) != manifest["provenance_bytes"]:
    raise ValueError(
        f"provenance.jsonl byte length mismatch: manifest specifies {manifest['provenance_bytes']}, "
        f"actual length is {len(raw_provenance_bytes)}"
    )
actual_provenance_sha256 = hashlib.sha256(raw_provenance_bytes).hexdigest()
if actual_provenance_sha256 != manifest["provenance_sha256"]:
    raise ValueError(
        f"provenance.jsonl SHA-256 mismatch: manifest specifies {manifest['provenance_sha256']}, "
        f"actual is {actual_provenance_sha256}"
    )

provenance_records = []
for line_no, raw_line in enumerate(raw_provenance_bytes.decode("utf-8").splitlines(), start=1):
    line = raw_line.strip()
    if not line:
        continue
    try:
        prov_obj = json.loads(line)
    except Exception as exc:
        raise ValueError(f"provenance.jsonl line {line_no}: invalid JSON: {exc}") from exc
    if not isinstance(prov_obj, dict):
        raise ValueError(f"provenance.jsonl line {line_no}: record must be a JSON object, got {type(prov_obj).__name__}")
    for field_name, expected_type in REQUIRED_PROVENANCE_FIELDS.items():
        if field_name not in prov_obj:
            raise ValueError(f"provenance.jsonl line {line_no}: missing required field: {field_name!r}")
        val = prov_obj[field_name]
        if isinstance(expected_type, tuple):
            if not isinstance(val, expected_type):
                type_names = ", ".join(t.__name__ for t in expected_type)
                raise ValueError(
                    f"provenance.jsonl line {line_no}: field {field_name!r} expected one of ({type_names}), got {type(val).__name__}"
                )
        else:
            if not isinstance(val, expected_type):
                raise ValueError(
                    f"provenance.jsonl line {line_no}: field {field_name!r} expected {expected_type.__name__}, got {type(val).__name__}"
                )

    if not prov_obj["pair_id"].strip():
        raise ValueError(f"provenance.jsonl line {line_no}: 'pair_id' must be a non-empty string")

    for uuid_field in ("generation_id", "evaluation_id", "document_id"):
        if not _is_valid_uuid(prov_obj[uuid_field]):
            raise ValueError(f"provenance.jsonl line {line_no}: field {uuid_field!r} must be a valid UUID string")

    if prov_obj["agent_id"] not in SUPPORTED_AGENTS:
        raise ValueError(
            f"provenance.jsonl line {line_no}: unsupported agent_id {prov_obj['agent_id']!r}; supported agents: {sorted(SUPPORTED_AGENTS)}"
        )
    if prov_obj["agent_id"] != manifest["agent_id"]:
        raise ValueError(
            f"provenance.jsonl line {line_no}: agent_id {prov_obj['agent_id']!r} does not match manifest agent_id {manifest['agent_id']!r}"
        )

    if not prov_obj["unit_key"].strip():
        raise ValueError(f"provenance.jsonl line {line_no}: 'unit_key' must be a non-empty string")
    if not prov_obj["model_name"].strip():
        raise ValueError(f"provenance.jsonl line {line_no}: 'model_name' must be a non-empty string")
    if not prov_obj["response_contract_key"].strip():
        raise ValueError(f"provenance.jsonl line {line_no}: 'response_contract_key' must be a non-empty string")
    if isinstance(prov_obj["response_contract_version"], bool) or prov_obj["response_contract_version"] <= 0:
        raise ValueError(f"provenance.jsonl line {line_no}: 'response_contract_version' must be a positive integer")

    if not isinstance(prov_obj["criterion_ids"], list) or not all(isinstance(c, str) and c.strip() for c in prov_obj["criterion_ids"]):
        raise ValueError(f"provenance.jsonl line {line_no}: 'criterion_ids' must be a list of non-empty strings")

    if not isinstance(prov_obj["reviewer_ids"], list) or not all(_is_valid_uuid(r) for r in prov_obj["reviewer_ids"]):
        raise ValueError(f"provenance.jsonl line {line_no}: 'reviewer_ids' must be a list of valid UUID strings")

    for hash_field in ("prompt_sha256", "response_sha256"):
        hval = prov_obj[hash_field]
        if not HEX64_RE.match(hval):
            raise ValueError(f"provenance.jsonl line {line_no}: {hash_field!r} must be a 64-char hexadecimal string")

    provenance_records.append(prov_obj)

seen_pair_ids: set[str] = set()
for idx, prov in enumerate(provenance_records, start=1):
    pid = prov["pair_id"]
    if pid in seen_pair_ids:
        raise ValueError(f"provenance.jsonl record {idx}: duplicate pair_id {pid!r}")
    seen_pair_ids.add(pid)

pairs = []
for line_no, raw_line in enumerate(raw_pairs_bytes.decode("utf-8").splitlines(), start=1):
    line = raw_line.strip()
    if not line:
        continue
    try:
        pair_obj = json.loads(line)
    except Exception as exc:
        raise ValueError(f"pairs.jsonl line {line_no}: invalid JSON: {exc}") from exc
    if not isinstance(pair_obj, dict):
        raise ValueError(f"pairs.jsonl line {line_no}: record must be a JSON object, got {type(pair_obj).__name__}")
    missing_keys = REQUIRED_PAIR_KEYS - pair_obj.keys()
    if missing_keys:
        raise ValueError(f"pairs.jsonl line {line_no}: missing required key(s) {sorted(missing_keys)}")
    for key in REQUIRED_PAIR_KEYS:
        val = pair_obj[key]
        if not isinstance(val, str) or not val.strip():
            raise ValueError(f"pairs.jsonl line {line_no}: key {key!r} must be a non-empty string")
    if pair_obj["chosen"] == pair_obj["rejected"]:
        raise ValueError(f"pairs.jsonl line {line_no}: 'chosen' and 'rejected' are identical; invalid preference pair")
    pairs.append(pair_obj)

if manifest["evaluation_count"] < 0:
    raise ValueError(f"Manifest reports negative evaluation_count: {manifest['evaluation_count']}")

if manifest["reviewer_count"] < 0:
    raise ValueError(f"Manifest reports negative reviewer_count: {manifest['reviewer_count']}")

if any(count < 0 for count in manifest["skipped_counts"].values()):
    raise ValueError(f"Manifest reports negative skipped_counts: {manifest['skipped_counts']}")

if len(provenance_records) != len(pairs):
    raise ValueError(
        f"Parsed provenance record count ({len(provenance_records)}) does not match "
        f"pair count ({len(pairs)})"
    )

if len(pairs) == 0:
    raise ValueError("Package contains 0 pairs -- pair_count must be greater than 0")

if manifest["pair_count"] <= 0:
    raise ValueError(f"Manifest reports pair_count={manifest['pair_count']} <= 0 -- nothing to train on")

if len(pairs) != manifest["pair_count"]:
    raise ValueError(
        f"Parsed pair count ({len(pairs)}) does not match manifest.pair_count ({manifest['pair_count']})"
    )

# Positional one-to-one association: provenance_records[i] is associated with pairs[i]
pair_provenance_records = list(zip(pairs, provenance_records, strict=True))

for idx, (pair_item, prov_item) in enumerate(pair_provenance_records, start=1):
    if prov_item["pair_id"] != prov_item["generation_id"]:
        raise ValueError(
            f"Record {idx}: pair_id ({prov_item['pair_id']!r}) must match generation_id ({prov_item['generation_id']!r})"
        )
    expected_prompt_sha = hashlib.sha256(pair_item["prompt"].encode("utf-8")).hexdigest()
    if prov_item["prompt_sha256"] != expected_prompt_sha:
        raise ValueError(
            f"Record {idx}: prompt_sha256 mismatch (expected {expected_prompt_sha}, got {prov_item['prompt_sha256']})"
        )
    expected_response_sha = hashlib.sha256(pair_item["rejected"].encode("utf-8")).hexdigest()
    if prov_item["response_sha256"] != expected_response_sha:
        raise ValueError(
            f"Record {idx}: response_sha256 mismatch (expected {expected_response_sha}, got {prov_item['response_sha256']})"
        )

print(
    f"Validated package: agent={manifest['agent_id']}, {len(pairs)} pairs, "
    f"{len(provenance_records)} provenance records. Package SHA-256 and structure verified."
)

## Training

The cells below install dependencies, hold out whole evaluations (split by
`evaluation_id`) from the fetched and verified `pairs`, load a 4-bit quantized
base model with a LoRA adapter, run TRL's `DPOTrainer`, print a sanity-check
evaluation on the held-out set (the real adapter-vs-base comparison is
`training/evaluate_adapter.py`), and bundle a reproducible provenance training
manifest plus the held-out pairs (`heldout_pairs.jsonl`) into the adapter before
the push-back cell zips and uploads it.

The LoRA rank/alpha and DPO hyperparameters below are a reasonable
starting recipe for a small model on Colab's free-tier GPU, not a tuned
result -- edit them directly in the cells if you have reason to.

In [ ]:
# Dependency pinning note:
# Pinned, primary-source-compatible pins for this Colab T4 workflow with fresh Colab compatibility revalidation pending:
#   unsloth==2025.3.10, trl==0.15.2, peft==0.14.0, bitsandbytes==0.45.3,
#   datasets==3.3.2, transformers==4.50.0, accelerate==1.4.0
# If running in a fresh Colab GPU runtime, set these explicitly to ensure exact
# reproducibility across library upgrades.
PIN_UNSLOTH = "==2025.3.10"
PIN_TRL = "==0.15.2"
PIN_PEFT = "==0.14.0"
PIN_BITSANDBYTES = "==0.45.3"
PIN_DATASETS = "==3.3.2"
PIN_TRANSFORMERS = "==4.50.0"
PIN_ACCELERATE = "==1.4.0"

install_pkgs = [
    f"unsloth{PIN_UNSLOTH}",
    f"trl{PIN_TRL}",
    f"peft{PIN_PEFT}",
    f"bitsandbytes{PIN_BITSANDBYTES}",
    f"datasets{PIN_DATASETS}",
    f"transformers{PIN_TRANSFORMERS}",
    f"accelerate{PIN_ACCELERATE}",
]
!pip install -q {" ".join(install_pkgs)}

In [ ]:
import hashlib

from datasets import Dataset

MIN_PAIRS_FOR_EVAL_SPLIT = 20
HELDOUT_PERCENT = 20
HELDOUT_SEED = 42
HELDOUT_METHOD = "group_by_evaluation_id"


def split_by_evaluation(pair_provenance, percent, seed):
    """Hold out whole evaluations so none is on both sides of the split.

    pair_provenance is a list of (pair, provenance_record). Evaluations are
    ordered by sha256("<seed>:<evaluation_id>") -- deterministic, no RNG -- and
    the first ceil(percent% of them) are held out: at least one, and never all
    of them. A single evaluation cannot be split, so nothing is held out then.
    Returns (train_items, heldout_items), each keeping the original order.
    """
    evaluation_ids = {prov["evaluation_id"] for _, prov in pair_provenance}
    if len(evaluation_ids) < 2:
        return list(pair_provenance), []
    ordered = sorted(
        evaluation_ids,
        key=lambda eid: hashlib.sha256(f"{seed}:{eid}".encode("utf-8")).hexdigest(),
    )
    count = (len(ordered) * percent + 99) // 100
    count = min(max(1, count), len(ordered) - 1)
    heldout_ids = set(ordered[:count])
    train = [item for item in pair_provenance if item[1]["evaluation_id"] not in heldout_ids]
    heldout = [item for item in pair_provenance if item[1]["evaluation_id"] in heldout_ids]
    return train, heldout


def _training_row(pair):
    return {key: pair[key] for key in ("prompt", "chosen", "rejected")}


if len(pairs) >= MIN_PAIRS_FOR_EVAL_SPLIT:
    train_items, heldout_items = split_by_evaluation(
        pair_provenance_records, HELDOUT_PERCENT, HELDOUT_SEED
    )
else:
    train_items, heldout_items = list(pair_provenance_records), []

train_dataset = Dataset.from_list([_training_row(pair) for pair, _ in train_items])
if heldout_items:
    eval_dataset = Dataset.from_list([_training_row(pair) for pair, _ in heldout_items])
    # Saved into the adapter zip by the manifest cell so the adapter can be
    # evaluated later on pairs it never trained on.
    heldout_rows = [
        {
            "pair_id": prov["pair_id"],
            "evaluation_id": prov["evaluation_id"],
            **_training_row(pair),
        }
        for pair, prov in heldout_items
    ]
else:
    eval_dataset = None
    heldout_rows = []
    reason = (
        f"only {len(pairs)} pairs are available (< {MIN_PAIRS_FOR_EVAL_SPLIT})"
        if len(pairs) < MIN_PAIRS_FOR_EVAL_SPLIT
        else "every pair comes from a single evaluation"
    )
    print(
        f"Holding nothing out: {reason}. Training on all pairs; the eval "
        "sanity-check cell below will be skipped and this adapter will carry "
        "no held-out set, so it cannot be evaluated later."
    )

print(
    f"train_dataset: {len(train_dataset)} pairs from "
    f"{len({prov['evaluation_id'] for _, prov in train_items})} evaluation(s)"
)
if eval_dataset is not None:
    print(
        f"held-out: {len(heldout_rows)} pairs from "
        f"{len({row['evaluation_id'] for row in heldout_rows})} evaluation(s), "
        "saved with the adapter so it can be evaluated later"
    )

In [ ]:
from unsloth import FastLanguageModel

# Model specification:
# Pinned base model and primary-source-compatible commit revision SHA:
BASE_MODEL_NAME = "unsloth/gemma-3-4b-it"
BASE_MODEL_REVISION = "21bc97b90507086e76f0d256fee672973085d905"
MAX_SEQ_LENGTH = 2048

model_load_kwargs = dict(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
if BASE_MODEL_REVISION:
    model_load_kwargs["revision"] = BASE_MODEL_REVISION

model, tokenizer = FastLanguageModel.from_pretrained(**model_load_kwargs)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

In [ ]:
from trl import DPOConfig, DPOTrainer
from unsloth import is_bfloat16_supported

ADAPTER_DIR = "./trained_adapter"
TRAINING_SEED = 42
USE_BF16 = bool(is_bfloat16_supported())
USE_FP16 = not USE_BF16
PRECISION_NAME = "bfloat16" if USE_BF16 else "float16"

training_args = DPOConfig(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_train_epochs=1,
    beta=0.1,
    seed=TRAINING_SEED,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=1536,
    eval_strategy="steps" if eval_dataset is not None else "no",
    eval_steps=20,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=USE_FP16,
    bf16=USE_BF16,
)
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
trainer.train()

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

In [ ]:
if eval_dataset is not None:
    try:
        metrics = trainer.evaluate()
        print(f"eval_loss: {metrics['eval_loss']:.4f}")
        print(
            "reward accuracy (chosen > rejected): "
            f"{metrics.get('eval_rewards/accuracies', 'n/a')}"
        )
        print(
            "NOTE: this is an in-run sanity check on the evaluations held out "
            "of THIS training run (split by evaluation_id). It does not "
            "compare against the base model. The held-out pairs are saved in "
            "the adapter zip as heldout_pairs.jsonl; run "
            "training/evaluate_adapter.py against the served adapter for the "
            "real adapter-vs-base comparison. A low accuracy here is a strong "
            "signal something went wrong; a high accuracy is not by itself a "
            "green light to deploy."
        )
    except Exception as exc:
        # This cell is a soft, non-blocking sanity check -- a crash here
        # (e.g. a known transformers/notebook-progress-callback quirk on
        # very short training runs) must never stop the notebook from
        # reaching the push-back cell below.
        metrics = None
        print(f"Eval sanity check failed to run ({exc!r}); skipping it. "
              "This does not affect the trained adapter -- continuing to "
              "the upload cell.")
else:
    metrics = None
    print("Skipped eval (nothing was held out -- see the split cell above).")

In [ ]:
import hashlib
import importlib.metadata
import json as _json
import os

def _get_pkg_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "unknown"

installed_dependency_versions = {
    pkg: _get_pkg_version(pkg)
    for pkg in ("unsloth", "trl", "peft", "bitsandbytes", "datasets", "transformers", "torch", "accelerate")
}

HELDOUT_FILENAME = "heldout_pairs.jsonl"
if heldout_rows:
    heldout_bytes = "".join(
        _json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n"
        for row in heldout_rows
    ).encode("utf-8")
    with open(os.path.join(ADAPTER_DIR, HELDOUT_FILENAME), "wb") as f:
        f.write(heldout_bytes)
    heldout_manifest = {
        "method": HELDOUT_METHOD,
        "seed": HELDOUT_SEED,
        "fraction": HELDOUT_PERCENT / 100,
        "pair_count": len(heldout_rows),
        "evaluation_count": len({row["evaluation_id"] for row in heldout_rows}),
        "sha256": hashlib.sha256(heldout_bytes).hexdigest(),
    }
else:
    heldout_manifest = None

training_manifest = {
    "source_job_manifest": manifest,
    "source_hashes": {
        "pairs_sha256": manifest.get("pairs_sha256"),
        "provenance_sha256": manifest.get("provenance_sha256"),
    },
    "base_model": {
        "name": BASE_MODEL_NAME,
        "revision": BASE_MODEL_REVISION,
    },
    "dependencies": installed_dependency_versions,
    "seed": training_args.seed if hasattr(training_args, "seed") else TRAINING_SEED,
    "precision": {
        "name": PRECISION_NAME,
        "fp16": USE_FP16,
        "bf16": USE_BF16,
    },
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_config": {
        k: v
        for k, v in model.peft_config["default"].to_dict().items()
        if k in ("r", "lora_alpha", "lora_dropout", "target_modules")
    },
    "training_args": {
        "learning_rate": training_args.learning_rate,
        "num_train_epochs": training_args.num_train_epochs,
        "beta": training_args.beta,
        "per_device_train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "seed": training_args.seed if hasattr(training_args, "seed") else TRAINING_SEED,
    },
    "pair_count": len(pairs),
    "heldout": heldout_manifest,
    "eval_metrics": metrics,
}
training_manifest["lora_config"]["target_modules"] = sorted(
    training_manifest["lora_config"]["target_modules"]
)
with open(os.path.join(ADAPTER_DIR, "training_manifest.json"), "w") as f:
    _json.dump(training_manifest, f, indent=2)

print(f"Wrote {ADAPTER_DIR}/training_manifest.json")

In [ ]:
import os
import zipfile as zf_module

import requests

ADAPTER_ZIP_PATH = "trained_adapter.zip"
REQUIRED_OUTPUT_FILES = frozenset({"adapter_config.json", "training_manifest.json"})

if not os.path.isdir(ADAPTER_DIR):
    raise FileNotFoundError(f"Adapter output directory does not exist: {ADAPTER_DIR}")

# Collect files deterministically
entries_to_archive: list[tuple[str, str]] = []
for root, dirs, files in os.walk(ADAPTER_DIR):
    dirs.sort()
    for name in sorted(files):
        full_path = os.path.join(root, name)
        rel_path = os.path.relpath(full_path, ADAPTER_DIR)
        # Forward slashes for zip internal entry consistency
        arcname = rel_path.replace(os.sep, "/")
        entries_to_archive.append((full_path, arcname))

entries_to_archive.sort(key=lambda x: x[1])
archived_names = frozenset(arcname for _, arcname in entries_to_archive)

# Validate required files before creating zip and consuming single-use upload URL
missing_required = REQUIRED_OUTPUT_FILES - archived_names
if missing_required:
    raise FileNotFoundError(
        f"Trained adapter directory is missing required output files: {sorted(missing_required)}"
    )

has_adapter_weights = any(
    name == "adapter_model.safetensors" or name == "adapter_model.bin"
    for name in archived_names
)
if not has_adapter_weights:
    raise FileNotFoundError(
        f"Trained adapter directory does not contain adapter_model.safetensors or adapter_model.bin. Found: {sorted(archived_names)}"
    )

with zf_module.ZipFile(ADAPTER_ZIP_PATH, mode="w", compression=zf_module.ZIP_DEFLATED) as zf:
    for full_path, arcname in entries_to_archive:
        zf.write(full_path, arcname=arcname)

print(f"Created {ADAPTER_ZIP_PATH} with {len(entries_to_archive)} entries (relative paths preserved).")

# POST to upload URL only after validating output files
with open(ADAPTER_ZIP_PATH, "rb") as f:
    upload_response = requests.post(
        UPLOAD_URL,
        files={"file": ("adapter.zip", f, "application/zip")},
    )
upload_response.raise_for_status()
print("Adapter uploaded:", upload_response.json())

## Convert the adapter to GGUF (after the upload)

The trained adapter has already been uploaded to EquipED by the cell above, so
nothing below can lose it. The next cells convert that same adapter into a GGUF
LoRA file that the host's llama.cpp `llama-server` can load, verify it, and offer
it for download. This adds roughly 5-10 minutes (it installs the llama.cpp
converter into a separate virtual environment, so the training libraries above
are not disturbed).

You get three files: `adapter-f16.gguf`, `adapter-f16.gguf.sha256` (its
checksum) and `adapter-f16.gguf.json` (which llama.cpp commit made it, and which
uploaded adapter it came from). Nothing is deployed: give the files to the host
owner, who loads them by hand (`training/serving-lora-adapter.md`).

If any step below fails, the adapter is still safe on the backend. Re-convert it
later with `docs/colab/adapter_to_gguf_template.ipynb`.

In [ ]:
import contextlib
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# The converter needs the regular instruct model's config, not the 4-bit variant
# named inside adapter_config.json; BASE_MODEL_NAME (cell 6) is exactly that.
GGUF_BASE_MODEL_ID = BASE_MODEL_NAME
OUTPUT_GGUF = "adapter-f16.gguf"
GGUF_OUTPUT_FILES = [OUTPUT_GGUF, OUTPUT_GGUF + ".sha256", OUTPUT_GGUF + ".json"]
LLAMA_CPP_REPO = "https://github.com/ggml-org/llama.cpp"
# None = llama.cpp's default branch. To match the host's llama-server build,
# set a FULL commit hash from that build's source tree.
LLAMA_CPP_REF = None
CONVERTER_VENV = "converter-venv"

# Runs inside the converter virtual environment: reads the GGUF with llama.cpp's
# own gguf package and prints what check_lora_fields needs as one JSON line.
GGUF_DUMP_SCRIPT = """
import json
import sys

sys.path.insert(0, "llama.cpp/gguf-py")
from gguf import GGUFReader


def plain(value):
    return value.tolist() if hasattr(value, "tolist") else value


reader = GGUFReader(sys.argv[1])


def field_value(name):
    field = reader.fields.get(name)
    return None if field is None else plain(field.contents())


print(
    json.dumps(
        {
            "general_type": field_value("general.type"),
            "adapter_type": field_value("adapter.type"),
            "alpha": field_value("adapter.lora.alpha"),
            "tensors": [
                [tensor.name, [int(dim) for dim in tensor.shape]]
                for tensor in reader.tensors
            ],
        }
    )
)
"""


def run(command):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    subprocess.run(command, check=True)


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def check_lora_fields(
    general_type, adapter_type, alpha, tensors, expected_rank, expected_alpha
):
    problems = []
    if general_type != "adapter":
        problems.append(f"general.type is {general_type!r}, expected 'adapter'")
    if adapter_type != "lora":
        problems.append(f"adapter.type is {adapter_type!r}, expected 'lora'")
    if alpha is None or abs(float(alpha) - expected_alpha) > 1e-6:
        problems.append(f"alpha is {alpha!r}, expected {expected_alpha}")
    lora_a = [item for item in tensors if item[0].endswith(".lora_a")]
    lora_b = [item for item in tensors if item[0].endswith(".lora_b")]
    if not lora_a:
        problems.append("no lora_a tensors found")
    if len(lora_a) != len(lora_b):
        problems.append(f"{len(lora_a)} lora_a tensors but {len(lora_b)} lora_b tensors")
    for name, shape in lora_a + lora_b:
        if min(shape) != expected_rank:
            problems.append(
                f"{name}: smallest dimension {min(shape)} is not rank {expected_rank}"
            )
            break
    if problems:
        raise ValueError("; ".join(problems))
    return {"tensor_pairs": len(lora_a), "rank": expected_rank, "alpha": float(alpha)}


@contextlib.contextmanager
def conversion_step(name):
    """Say the adapter is safe if a conversion step fails, then re-raise."""
    try:
        yield
    except Exception:
        print(
            f"\nGGUF conversion step '{name}' failed. The trained adapter was "
            "already uploaded to EquipED and is safe. Re-convert it later with "
            "docs/colab/adapter_to_gguf_template.ipynb."
        )
        raise


def venv_python():
    """The interpreter inside the converter virtual environment."""
    scripts, exe = ("Scripts", "python.exe") if os.name == "nt" else ("bin", "python")
    return str(Path(CONVERTER_VENV).resolve() / scripts / exe)


def read_gguf_fields(gguf_path, python=None):
    """Read the GGUF with llama.cpp's gguf package, inside the converter venv."""
    result = subprocess.run(
        [python or venv_python(), "-c", GGUF_DUMP_SCRIPT, str(gguf_path)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"could not read {gguf_path} (exit {result.returncode})")
    return json.loads(result.stdout.strip().splitlines()[-1])


def build_gguf_metadata(
    *,
    llama_cpp_commit,
    llama_cpp_ref,
    base_model_id,
    lora_rank,
    lora_alpha,
    tensor_pairs,
    gguf_sha256,
    gguf_bytes,
    adapter_zip_sha256,
):
    return {
        "llama_cpp_commit": llama_cpp_commit,
        "llama_cpp_ref": llama_cpp_ref,
        "converter": "convert_lora_to_gguf.py",
        "outtype": "f16",
        "base_model_id": base_model_id,
        "lora_rank": lora_rank,
        "lora_alpha": lora_alpha,
        "tensor_pairs": tensor_pairs,
        "gguf_sha256": gguf_sha256,
        "gguf_bytes": gguf_bytes,
        "adapter_zip_sha256": adapter_zip_sha256,
    }

In [ ]:
with conversion_step("read the saved adapter"):
    # Best effort: give the converter as much RAM as possible.
    try:
        import gc

        for _name in ("trainer", "model"):
            globals().pop(_name, None)
        gc.collect()
        import torch

        torch.cuda.empty_cache()
    except Exception as exc:
        print(f"(could not free memory, continuing: {exc!r})")

    adapter_config = json.loads(
        (Path(ADAPTER_DIR) / "adapter_config.json").read_text(encoding="utf-8")
    )
    if adapter_config.get("peft_type") != "LORA":
        raise ValueError(
            f"not a LoRA adapter: peft_type={adapter_config.get('peft_type')!r}"
        )
    if not (Path(ADAPTER_DIR) / "adapter_model.safetensors").exists():
        raise FileNotFoundError("adapter_model.safetensors is missing from the adapter")
    LORA_RANK = int(adapter_config["r"])
    LORA_ALPHA = float(adapter_config["lora_alpha"])
    print("adapter base (as trained):", adapter_config.get("base_model_name_or_path"))
    print("converter will use base config from:", GGUF_BASE_MODEL_ID)
    print("rank:", LORA_RANK, "alpha:", LORA_ALPHA)

In [ ]:
with conversion_step("prepare the converter"):
    if not os.path.isdir("llama.cpp"):
        run(["git", "clone", "--depth", "1", LLAMA_CPP_REPO, "llama.cpp"])
        if LLAMA_CPP_REF:
            fetched = subprocess.run(
                ["git", "-C", "llama.cpp", "fetch", "--depth", "1", "origin", LLAMA_CPP_REF]
            )
            if fetched.returncode == 0:
                run(["git", "-C", "llama.cpp", "checkout", "FETCH_HEAD"])
            else:
                print(f"WARNING: could not fetch {LLAMA_CPP_REF}; using the default branch.")
    LLAMA_CPP_COMMIT = subprocess.run(
        ["git", "-C", "llama.cpp", "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print("llama.cpp commit:", LLAMA_CPP_COMMIT)

    requirements_dir = Path("llama.cpp/requirements")
    candidates = [
        requirements_dir / "requirements-convert_lora_to_gguf.txt",
        requirements_dir / "requirements-convert_hf_to_gguf.txt",
    ]
    requirements_file = next((path for path in candidates if path.exists()), None)
    if requirements_file is None:
        raise FileNotFoundError(
            "no converter requirements file under llama.cpp/requirements; "
            "the llama.cpp layout may have changed, check that directory"
        )

    # A separate virtual environment keeps the converter's packages away from
    # the pinned training stack. Some Colab images lack python's venv support
    # (ensurepip), so fall back to virtualenv.
    shutil.rmtree(CONVERTER_VENV, ignore_errors=True)
    try:
        run([sys.executable, "-m", "venv", CONVERTER_VENV])
    except subprocess.CalledProcessError:
        print("python -m venv failed; falling back to virtualenv")
        shutil.rmtree(CONVERTER_VENV, ignore_errors=True)
        run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        run([sys.executable, "-m", "virtualenv", CONVERTER_VENV])
    run([venv_python(), "-m", "pip", "install", "-q", "-r", requirements_file])

In [ ]:
with conversion_step("convert to GGUF"):
    # Print the converter's own flags first, so the log records what this
    # llama.cpp version actually accepts.
    run([venv_python(), "llama.cpp/convert_lora_to_gguf.py", "--help"])

    run(
        [
            venv_python(),
            "llama.cpp/convert_lora_to_gguf.py",
            "--base-model-id", GGUF_BASE_MODEL_ID,
            "--outfile", OUTPUT_GGUF,
            "--outtype", "f16",
            ADAPTER_DIR,
        ]
    )
    if not os.path.exists(OUTPUT_GGUF):
        raise FileNotFoundError(f"converter finished but {OUTPUT_GGUF!r} was not created")

In [ ]:
with conversion_step("verify the GGUF"):
    fields = read_gguf_fields(OUTPUT_GGUF)
    tensors = [(name, tuple(shape)) for name, shape in fields["tensors"]]
    summary = check_lora_fields(
        fields["general_type"],
        fields["adapter_type"],
        fields["alpha"],
        tensors,
        LORA_RANK,
        LORA_ALPHA,
    )
    print("GGUF LoRA verified:", summary)

    gguf_sha256 = sha256_of(OUTPUT_GGUF)
    gguf_bytes = os.path.getsize(OUTPUT_GGUF)
    Path(OUTPUT_GGUF + ".sha256").write_text(
        f"{gguf_sha256}  {OUTPUT_GGUF}\n", encoding="utf-8"
    )
    gguf_metadata = build_gguf_metadata(
        llama_cpp_commit=LLAMA_CPP_COMMIT,
        llama_cpp_ref=LLAMA_CPP_REF,
        base_model_id=GGUF_BASE_MODEL_ID,
        lora_rank=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        tensor_pairs=summary["tensor_pairs"],
        gguf_sha256=gguf_sha256,
        gguf_bytes=gguf_bytes,
        adapter_zip_sha256=sha256_of(ADAPTER_ZIP_PATH),
    )
    Path(OUTPUT_GGUF + ".json").write_text(
        json.dumps(gguf_metadata, indent=2) + "\n", encoding="utf-8"
    )
    print(f"{OUTPUT_GGUF}: {gguf_bytes / (1024 * 1024):.1f} MB")
    print("sha256:", gguf_sha256)
    print("llama.cpp commit:", LLAMA_CPP_COMMIT)

In [ ]:
with conversion_step("download the files"):
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab: the files are in the working directory:")
        print(", ".join(GGUF_OUTPUT_FILES))
    else:
        print("Your browser may ask to allow multiple downloads; allow them.")
        for _name in GGUF_OUTPUT_FILES:
            files.download(_name)
        print("Give these files to the host owner (training/serving-lora-adapter.md).")